# **ML Model:** XGBoost

## **Notes:**

**XGBoost** (*eXtreme Gradient Boosting*) is boosting algorithm. Unlike **RF** (*Random Forest*) which builds trees parallel, boosting builds them sequentially and each of them tries to correct bad decisions of the previous one.

## **Implementation:**

#### Librabry imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score

### **Load data**

In [2]:
data = pd.read_excel('../data/processed/accident_processed_srb.xlsx')
data.head(3)

,longitude,latitude,accident_type,municipality_encoded,month,day_of_week,hour,day_type,is_rush,is_night,season,acc_parked_vehicles,acc_pedestrians,acc_single_vehicle,acc_two_vehicles_no_turn,acc_two_vehicles_turn_or_cross,description_encoded
0,20.301589,44.568563,0,0.698704,1,1,10,1,0,0,0,0,0,1,0,0,0.541965
1,20.413280,44.579780,0,0.698704,1,3,12,1,0,0,0,0,0,0,1,0,0.276762
2,20.312560,44.575470,0,0.698704,1,4,10,1,0,0,0,0,0,0,0,1,0.610821


In [3]:
X = data.drop(columns='accident_type')
y = data['accident_type']

In [4]:
X.head(3)

,longitude,latitude,municipality_encoded,month,day_of_week,hour,day_type,is_rush,is_night,season,acc_parked_vehicles,acc_pedestrians,acc_single_vehicle,acc_two_vehicles_no_turn,acc_two_vehicles_turn_or_cross,description_encoded
0,20.301589,44.568563,0.698704,1,1,10,1,0,0,0,0,0,1,0,0,0.541965
1,20.413280,44.579780,0.698704,1,3,12,1,0,0,0,0,0,0,1,0,0.276762
2,20.312560,44.575470,0.698704,1,4,10,1,0,0,0,0,0,0,0,1,0.610821


In [5]:
y.head(3)

0    0
1    0
2    0
Name: accident_type, dtype: int64

### **Split data**

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

### **Training**

In [8]:
import xgboost as xgb

In [9]:
xgb_model = xgb.XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    reg_alpha=0.0,
    objective='multi:softmax',
    num_class=len(y.unique()),
    eval_metric='mlogloss',
    early_stopping_rounds=50,
    random_state=42,
    n_jobs=-1
)

In [ ]:
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=50
)

print(f"\nOptimalan broj stabala: {xgb_model.best_iteration}")

[0]	validation_0-mlogloss:0.66099	validation_1-mlogloss:0.66091
[50]	validation_0-mlogloss:0.48494	validation_1-mlogloss:0.48623
[100]	validation_0-mlogloss:0.47447	validation_1-mlogloss:0.47809
[150]	validation_0-mlogloss:0.46948	validation_1-mlogloss:0.47621
[200]	validation_0-mlogloss:0.46568	validation_1-mlogloss:0.47535
[250]	validation_0-mlogloss:0.46233	validation_1-mlogloss:0.47480
[300]	validation_0-mlogloss:0.45919	validation_1-mlogloss:0.47445
[350]	validation_0-mlogloss:0.45625	validation_1-mlogloss:0.47423
[400]	validation_0-mlogloss:0.45344	validation_1-mlogloss:0.47408
[450]	validation_0-mlogloss:0.45071	validation_1-mlogloss:0.47406
[500]	validation_0-mlogloss:0.44816	validation_1-mlogloss:0.47399
[550]	validation_0-mlogloss:0.44564	validation_1-mlogloss:0.47401
[560]	validation_0-mlogloss:0.44520	validation_1-mlogloss:0.47399

Optimalan broj stabala: 510


### **Evaluation**

In [ ]:
y_pred = xgb_model.predict(X_test)

In [ ]:
print("Classification Report:")
print(classification_report(y_test, y_pred))

Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.83      0.80     24509
           1       0.72      0.65      0.68     16562

    accuracy                           0.76     41071
   macro avg       0.75      0.74      0.74     41071
weighted avg       0.76      0.76      0.76     41071



In [ ]:
f1 = f1_score(y_test, y_pred, average='weighted')
print(f"Weighted F1 Score: {f1:.4f}")

Weighted F1 Score: 0.7553


### **Additional:**